[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/thermodynamics/contact_geometry/contact_geometry.ipynb)

# Legendre Transformation and Contact Geometry

This example computes the Legendre change from a gas's energy as a function of its entropy to its Helmholtz energy as a function of its temperature. Holding the temperature is easier in the laboratory than holding the entropy, and the Helmholtz energy, the energy less temperature times entropy, is the potential that describes the gas then. Geometrically, each tangent line of the energy's graph has the temperature as its slope and the Helmholtz energy as its height at zero entropy, and the graph of the Helmholtz energy is traced by those heights.

The five quantities of the gas are modelled as the readings of an antivector of a five-dimensional space, `mv.t & point`, and equilibrium as a surface in it, each state of entropy and volume lifted to the energy, temperature and pressure the energy function gives there. Along the surface the contact reading, `contact_element(point) & change`, vanishes for every change: a change of energy is the heat less the work. The Legendre change is a map of the whole space, and its derivative carries changes; the contact reading at the new point, applied to the carried changes, is the contact reading at the old point, so the change takes equilibrium surfaces to equilibrium surfaces. The contact reading's own derivative, `mv.pv - mv.ts`, wedged twice with the contact element, is a nonzero five-blade: the contact planes twist everywhere, so the only surfaces lying in them are lifted ones like the equilibrium surface.

In the notation of differential forms, the contact form is $\theta = dU - T\,dS + p\,dV$, equilibrium is a Legendrian surface on which $\theta$ vanishes, and $\theta \wedge d\theta \wedge d\theta \neq 0$. The Helmholtz energy is $F = U - TS$, with $dF = -S\,dT - p\,dV$.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples,jax] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
import numpy as np

from numga import Algebra
from numga.backend.jax import JaxContext, derivative
from examples.util.animation import show_animation
from examples.thermodynamics.contact_geometry import render

ga = Algebra("u0s0v0t0p0")
mv = JaxContext(ga, dtype=np.float64).multivector
Scalar = ga.gatype.scalar()                                         # Scalar+
Ambient = ga.gatype.antivector()                                    # Antivector-
# The states of entropy and volume, a subspace of the five quantities.
State = (mv.s.dual() + mv.v.dual()).gatype.derive.output            # Antivector-
ContactElement = ga.gatype.vector()                                 # Vector+

# The gas's heat capacity at fixed volume; entropy is in nR, volume and temperature in reference values, and energy in
# nRT₀.
heat_capacity = 1.5

## Equilibrium in five quantities

The energy of the ideal gas is a function of its entropy and volume; its derivative there gives the temperature and minus the pressure, and the state lifted with all three is a point of equilibrium. The tangents of a section at fixed volume are the derivative of the lift bound to a step of entropy, and the contact reading of every one of them vanishes. The chart shows the energy against the entropy at several volumes, green, the section at unit volume bold, with grey-blue tangent lines enveloping it and one tangent orange, which crosses zero entropy at the Helmholtz energy.

In [ ]:
# A grid of entropies at several volumes for the surface, and the volume of the section followed.
n_entropies, section_volume = 41, 1.0
entropies, volumes = np.linspace(-0.8, 1.2, n_entropies), np.array([0.6, 0.8, 1.0, 1.3, 1.7])

In [ ]:
def internal_energy(state: State) -> Scalar:
    """The energy of an ideal gas with a constant heat capacity at fixed volume."""
    return heat_capacity * (((mv.s & state) - (mv.v & state).log()) / heat_capacity).exp()

def equilibrium(state: State) -> Ambient:
    """Each state lifted to its five quantities: the energy, and its slopes along the entropy and the volume, the
    temperature and minus the pressure."""
    differential = derivative(internal_energy)(state)                               # [...] Scalar+ <- State-
    return (state + mv.u.dual() * internal_energy(state) + mv.t.dual() * differential(mv.s.dual())
            - mv.p.dual() * differential(mv.v.dual()))

def contact_element(point: Ambient) -> ContactElement:
    """The vector whose pairing with a change vanishes when the change of energy is the heat less the work."""
    return mv.u - (mv.t & point) * mv.s + (mv.p & point) * mv.v

family_states = mv.s.dual() * entropies[None, :] + mv.v.dual() * volumes[:, None]           # [volumes, entropies] State-
family = equilibrium(family_states)                                                          # [volumes, entropies] Ambient-
section_states = mv.s.dual() * entropies + mv.v.dual() * section_volume                      # [entropies] State-
points = equilibrium(section_states)                                                         # [entropies] Ambient-
tangents = derivative(equilibrium)(section_states)(mv.s.dual())                              # [entropies] Ambient-
render.draw_energy(family, points, tangents);

## The Legendre change

The change to the Helmholtz energy exchanges entropy and temperature, with a sign so the contact reading keeps its form, and puts the energy less temperature times entropy in place of the energy. Its derivative at each point carries that point's tangent to the transformed section, and the contact reading at the transformed point, composed with the derivative, is the contact reading at the original point: `contact_element(helmholtz(point)) & derivative(helmholtz)(point)`. On the right, the transformed section is the Helmholtz energy against the temperature, its tangent's slope is minus the entropy, and its height at zero temperature is the energy again.

In [ ]:
def helmholtz(point: Ambient) -> Ambient:
    """The Legendre change to the Helmholtz energy, exchanging entropy and temperature."""
    entropy, temperature = mv.s & point, mv.t & point
    return (mv.u.dual() * ((mv.u & point) - temperature * entropy) + mv.s.dual() * temperature - mv.t.dual() * entropy
            + mv.v.dual() * (mv.v & point) + mv.p.dual() * (mv.p & point))

changed_family = helmholtz(family)                                                           # [volumes, entropies] Ambient-
changed_points = helmholtz(points)                                                           # [entropies] Ambient-
helmholtz_derivative = derivative(helmholtz)(points)                                         # [entropies] Ambient- <- Ambient-
changed_tangents = helmholtz_derivative(tangents)                                            # [entropies] Ambient-
render.draw(family, changed_family, points, tangents, changed_points, changed_tangents);

## The moving tangent

The point of tangency runs along the section. On the left its tangent turns steeper as the gas warms, and its height at zero entropy, the Helmholtz energy, traces the curve on the right; there the tangent's slope is minus the entropy, and its height at zero temperature gives back the energy.

In [ ]:
def moving():
    """The point of tangency and its tangent at each sample of the section, before and after the change."""
    for index in range(points.shape[0]):
        yield points[index], tangents[index], changed_points[index], changed_tangents[index]

show_animation(render.legendre(family, changed_family, points, tangents, changed_points, changed_tangents, moving()),
               "thermodynamics_legendre", duration_ms=90)

In [ ]:
# checks
np.testing.assert_allclose((contact_element(points) & tangents).kernel, 0, atol=1e-10)
np.testing.assert_allclose((contact_element(changed_points) & changed_tangents).kernel, 0, atol=1e-10)
# The contact reading at the transformed points, pulled back through the change, is the contact reading at the points.
pulled_back = contact_element(changed_points) & helmholtz_derivative                         # [entropies] Scalar+ <- Ambient-
np.testing.assert_allclose((pulled_back - (contact_element(points) & Ambient)).kernel, 0, atol=1e-10)
# The Helmholtz energy's slope along the temperature is minus the entropy.
np.testing.assert_allclose(((mv.u & changed_tangents) / (mv.s & changed_tangents) + (mv.s & points)).kernel, 0, atol=1e-10)
# The contact reading's derivative, wedged twice with the contact element, fills the space.
contact_derivative = mv.pv - mv.ts                                                           # [] Bivector+
assert np.all(np.abs((contact_element(points) ^ contact_derivative ^ contact_derivative).kernel) > 1)